Import primary library

In [278]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from IPython.display import display
import json

np.set_printoptions(
    precision=4,      # 4 chữ số thập phân
    suppress=True,    # không dùng dạng 1.23e-05
    linewidth=200     # tránh xuống dòng quá sớm
)

In [279]:
import os
import kagglehub
import shutil
from sklearn.feature_extraction.text import CountVectorizer

Load Data

In [280]:
path = kagglehub.dataset_download(
    "arashnic/mind-news-dataset"
)

print("Dataset downloaded to: ", path)

Dataset downloaded to:  C:\Users\tonmi\.cache\kagglehub\datasets\arashnic\mind-news-dataset\versions\2


Copy data to working dir

In [281]:
source = os.path.join(path, "MINDsmall_train")

destination = r"D:\CDNC\MIND-research\data\raw"

os.makedirs(destination, exist_ok=True)

files = [
    "news.tsv",
    "behaviors.tsv",
    "entity_embedding.vec",
    "relation_embedding.vec",
]

for file in files:
    shutil.copy2(
        os.path.join(source, file),
        os.path.join(destination, file)
    )

print("Done")



Done


Get split-dataset - 300 line - News_ID + Category + News title

In [282]:
news_path = os.path.join(destination, "news.tsv")

if not os.path.exists(news_path):
    f_news_small = open(news_path, "x", encoding="utf-8")


columns = [
    "News_ID",
    "Category",
    "SubCategory",
    "Title",
    "Abstract",
    "URL",
    "Title_Entities",
    "Abstract_Entities"
]

news = pd.read_csv(
    news_path,
    sep="\t",
    names=columns,
    
)

news = news[["News_ID", "Category", "Title"]]

sample = news.sample(
    n=300,
    random_state=42
).reset_index(drop=True)

sample = news.head(300)
sample_dir = r"D:\CDNC\MIND-research\data\sample"

sample.to_csv(
    os.path.join(sample_dir, "news.csv"),
    index=False
)

print(sample.head())
print("Sample shape: ", sample.shape)

  News_ID   Category                                              Title
0  N55528  lifestyle  The Brands Queen Elizabeth, Prince Charles, an...
1  N19639     health                      50 Worst Habits For Belly Fat
2  N61837       news  The Cost of Trump's Aid Freeze in the Trenches...
3  N53526     health  I Was An NBA Wife. Here's How It Affected My M...
4  N38324     health  How to Get Rid of Skin Tags, According to a De...
Sample shape:  (300, 3)


In [283]:
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic

class Models:
    

    def __init__(self):
        self.sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

        self.svectorizer = CountVectorizer(
                stop_words="english",
            )

        self.topic_model = BERTopic(
            calculate_probabilities=True, # Important to set this to True for probability calculations
            verbose=True,
            vectorizer_model=self.svectorizer
        )

models = Models()

class VectorContext:
    def __init__(self, title_list):
        self.title_list = title_list
        self.semantic_vector_list = None
        self.probabilities_list = None
        self.topics_vector_list = None

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

### Vector Factory

`VectorFactory` được sử dụng để tạo đối tượng biểu diễn vector tương ứng với từng mô hình thông qua một giao diện thống nhất. Thay vì khởi tạo trực tiếp từng lớp, người dùng chỉ cần chỉ định loại mô hình (`sentence` hoặc `bertopic`), Factory sẽ trả về đối tượng phù hợp.

Thiết kế này giúp:
- Tách biệt logic khởi tạo khỏi logic xử lý.
- Dễ dàng thay thế hoặc mở rộng sang các mô hình mới mà không ảnh hưởng đến mã nguồn hiện có.
- Tăng khả năng bảo trì và tái sử dụng mã nguồn.

Vector Interface

In [284]:
from abc import ABC, abstractmethod

class Vector(ABC):
    @abstractmethod
    def get_vector(self):
        pass
    
    @abstractmethod
    def overview(self):
        pass

    @abstractmethod
    def summary(self):
        pass

Semantic vector

In [285]:
class SentenceVector(Vector):
    def __init__(self, title_list, model):
        self.title_list = title_list
        self.model = model
        self.semantic_vector = None

    def get_vector(self):
        self.semantic_vector = self.model.encode(
            self.title_list,
            show_progress_bar=True,
            convert_to_numpy=True
        )
        return self.semantic_vector
    
    def overview(self):
        print("=" * 60)
        print("Sentence Embedding Overview")
        print(f"[Sentence is: {self.title_list[0]}]")
        print("=" * 60)

        print(f"Documents : {len(self.semantic_vector)}")
        print(f"Dimension : {self.semantic_vector.shape[1]}")
        print(f"Shape     : {self.semantic_vector.shape}")

        print("\nFirst vector (first 10 values):")
        print(self.semantic_vector[0][:10], "...")

    def summary(self, sample_index=0):

        metrics = pd.DataFrame({
            "Metric": [
                "Embedding Model",
                "Number of Documents",
                "Embedding Dimension",
                "Output Shape"
            ],
            "Value": [
                self.model.__class__.__name__,
                len(self.semantic_vector),
                self.semantic_vector.shape[1],
                str(self.semantic_vector.shape)
            ]
        })

        # Format vector đẹp hơn
        vector = ", ".join(
            f"{x:.4f}" for x in self.semantic_vector[sample_index][:10]
        ) + ", ..."

        example = pd.DataFrame({
            "Sample Title": [
                self.title_list[sample_index]
            ],
            "Embedding (first 10 dims)": [
                f"[{vector}]"
            ]
        })

        return metrics, example


Topic vector

In [286]:
class BERTopicVector(Vector):
    def __init__(self, title_list, model):
        self.title_list = title_list
        self.model = model

        self.notice = "BERTopic depends on the sentence transformer model." \
        " Please ensure that the sentence transformer model is trained before using BERTopic."

        self.probabilities = None
        self.topics_vector = None

    def get_vector(self, semantic_vector=None):
        if semantic_vector is None:
            print(self.notice)
            return
        
        topics, probabilities = self.model.fit_transform(
                                    self.title_list,
                                    semantic_vector,
                                )
        self.probabilities = probabilities
        self.topics_vector = topics
        return self.topics_vector, self.probabilities
    
    def overview(self):
        print("=" * 60)
        print("BERTopic Overview")
        print(f"[Sentence is: {self.title_list[0]}]")
        print("=" * 60)

        print(f"Number of topics : {len(set(self.topics_vector))}")

        print("\nTopic distribution:")
        print(self.model.get_topic_info()[["Topic", "Count"]])

        print("\nFirst 5 document topics:")
        print(self.topics_vector[:5])

        print("\nProbability shape:")
        print(self.probabilities.shape)

        print("\nFirst 10 document probability:")
        print(self.probabilities[:10])

    
    def summary(self, sample_index=0):
        metrics_df = pd.DataFrame({
            "Metric": [
                "Topic Model",
                "Number of Documents",
                "Number of Topics",
                "Number of Outliers",
                "Probability Shape"
            ],
            "Value": [
                self.model.__class__.__name__,
                len(self.title_list),
                len(set(self.topics_vector) - {-1}),
                np.sum(np.array(self.topics_vector) == -1),
                str(self.probabilities.shape)
            ]
        })

        # =========================
        # 2. Topic Information
        # =========================
        topic_df = self.model.get_topic_info()[["Topic", "Count"]].copy()

        keywords = []

        for topic in topic_df["Topic"]:

            if topic == -1:
                keywords.append("Outlier")
            else:
                words = [
                    word
                    for word, _ in self.model.get_topic(topic)[:5]
                ]
                keywords.append(", ".join(words))

        topic_df["Top Keywords"] = keywords

        # =========================
        # 3. Example
        # =========================
        probs = ", ".join(
            f"{p:.4f}" for p in self.probabilities[sample_index]
        )

        sample_df = pd.DataFrame({
            "Sample Title": [
                self.title_list[sample_index]
            ],
            "Assigned Topic": [
                self.topics_vector[sample_index]
            ],
            "Probability Distribution": [
                f"[{probs}]"
            ]
        })

        return metrics_df, topic_df, sample_df

Title list

In [287]:
model_context = VectorContext(sample["Title"].tolist())

Semantic vector

In [288]:
sentence_vector= SentenceVector(model_context.title_list, models.sentence_model)

model_context.semantic_vector_list = sentence_vector.get_vector()
sentence_vector.overview()

metric, example = sentence_vector.summary()

print("\nSummary of Sentence Embedding:")
display(metric)
print("\nExample of Sentence Embedding:")
display(example)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Sentence Embedding Overview
[Sentence is: The Brands Queen Elizabeth, Prince Charles, and Prince Philip Swear By]
Documents : 300
Dimension : 384
Shape     : (300, 384)

First vector (first 10 values):
[-0.0093  0.0424  0.059   0.0121  0.0334  0.0196  0.0274 -0.0615 -0.0362 -0.039 ] ...

Summary of Sentence Embedding:


,Metric,Value
0,Embedding Model,SentenceTransformer
1,Number of Documents,300
2,Embedding Dimension,384
3,Output Shape,"(300, 384)"



Example of Sentence Embedding:


,Sample Title,Embedding (first 10 dims)
0,"The Brands Queen Elizabeth, Prince Charles, an...","[-0.0093, 0.0424, 0.0590, 0.0121, 0.0334, 0.01..."


In [289]:
bertopic_vector = BERTopicVector(model_context.title_list, models.topic_model)
topics_vector, probabilities = bertopic_vector.get_vector(model_context.semantic_vector_list)
model_context.probabilities_list = probabilities
model_context.topics_vector_list = topics_vector
bertopic_vector.overview()

metric, topic, example = bertopic_vector.summary()
print("\nSummary of BERTopic:")
display(metric)
print("\nTopic Information:")
display(topic)
print("\nExample of BERTopic:")
display(example)

2026-07-12 20:39:32,569 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-07-12 20:39:32,737 - BERTopic - Dimensionality - Completed ✓
2026-07-12 20:39:32,738 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-12 20:39:32,760 - BERTopic - Cluster - Completed ✓
2026-07-12 20:39:32,763 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-12 20:39:32,781 - BERTopic - Representation - Completed ✓


BERTopic Overview
[Sentence is: The Brands Queen Elizabeth, Prince Charles, and Prince Philip Swear By]
Number of topics : 7

Topic distribution:
   Topic  Count
0     -1     42
1      0    109
2      1     43
3      2     37
4      3     35
5      4     20
6      5     14

First 5 document topics:
[4, 0, 1, 0, 0]

Probability shape:
(300, 6)

First 10 document probability:
[[0.     0.     0.     0.     1.     0.    ]
 [0.3411 0.1596 0.0776 0.1765 0.1117 0.1334]
 [0.0472 0.7367 0.0468 0.0689 0.0392 0.0612]
 [0.2386 0.1457 0.0787 0.1892 0.129  0.1708]
 [0.27   0.1563 0.0775 0.1857 0.1199 0.1428]
 [0.0239 0.0492 0.7716 0.0414 0.0375 0.0408]
 [0.0453 0.1103 0.0526 0.1249 0.0485 0.0665]
 [0.0996 0.5227 0.0503 0.1462 0.0534 0.0695]
 [0.0865 0.0911 0.0593 0.3788 0.0946 0.1192]
 [0.1001 0.3002 0.0928 0.2122 0.0827 0.1142]]

Summary of BERTopic:


,Metric,Value
0,Topic Model,BERTopic
1,Number of Documents,300
2,Number of Topics,6
3,Number of Outliers,42
4,Probability Shape,"(300, 6)"



Topic Information:


,Topic,Count,Top Keywords
0,-1,42,Outlier
1,0,109,"things, make, 50, need, ideas"
2,1,43,"police, house, killed, 2020, clinton"
3,2,37,"nfl, football, week, season, hold"
4,3,35,"vegas, las, tv, time, 2021"
5,4,20,"royal, queen, prince, kate, family"
6,5,14,"star, celebs, stars, tracks, celebrities"



Example of BERTopic:


,Sample Title,Assigned Topic,Probability Distribution
0,"The Brands Queen Elizabeth, Prince Charles, an...",4,"[0.0000, 0.0000, 0.0000, 0.0000, 1.0000, 0.0000]"


Represented vector

In [296]:
class RepresentedVector(Vector):
    def __init__(self, title_list, sentence_vector_list, bertopic_vector_list):
        self.title_list = title_list
        self.sentence_vector_list = sentence_vector_list
        self.bertopic_vector_list = bertopic_vector_list
        self.represented_vector = {}

        
    def get_vector(self):
        for i in range(len(self.title_list)):
            self.represented_vector[i] = {
                                            "semantic": self.sentence_vector_list[i],
                                            "topic_distribution": self.bertopic_vector_list[i],
                                        }
        
        return self.represented_vector

    def overview(self):
        print("=" * 80)
        print("Represented Vector Overview")
        print("=" * 80)

        print(f"Documents           : {len(self.represented_vector)}")

        sample = self.represented_vector[0]

        print(f"Semantic Dimension  : {len(sample['semantic'])}")
        print(f"Topic Distribution  : {len(sample['topic_distribution'])}")
        print(f"Stored Fields       : {list(sample.keys())}")

        print("=" * 80)

    def summary(self, sample_index=0, preview_dims=4):

        semantic = self.sentence_vector_list[sample_index]

        semantic_preview = (
            [round(float(x), 4) for x in semantic[:preview_dims]]
            + ["..."]
            + [round(float(x), 4) for x in semantic[-preview_dims:]]
        )

        represented_vector = {
            "semantic_embedding": semantic_preview,
            "topic_probability": [
                round(float(p), 4)
                for p in self.bertopic_vector_list[sample_index]
            ]
        }

        print("Represented Vector:")
        print(json.dumps(represented_vector, indent=4))

        print("\nVector Description:")
        print(f"  • Semantic Embedding Dimension : {len(self.sentence_vector_list[0])}")
        print(f"  • Topic Probability Shape      : {len(self.bertopic_vector_list[0])}")

In [297]:
represented_vector = RepresentedVector(sample["Title"].tolist(), model_context.semantic_vector_list, model_context.probabilities_list)

represented_vector.get_vector()
represented_vector.overview()
represented_vector.summary(sample_index=0, preview_dims=4)

Represented Vector Overview
Documents           : 300
Semantic Dimension  : 384
Topic Distribution  : 6
Stored Fields       : ['semantic', 'topic_distribution']
Represented Vector:
{
    "semantic_embedding": [
        -0.0093,
        0.0424,
        0.059,
        0.0121,
        "...",
        0.0517,
        -0.1316,
        0.0508,
        -0.0301
    ],
    "topic_probability": [
        0.0,
        0.0,
        0.0,
        0.0,
        1.0,
        0.0
    ]
}

Vector Description:
  • Semantic Embedding Dimension : 384
  • Topic Probability Shape      : 6
